In [ ]:
from pathlib import Path
import sys
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display

# 저장소 루트와 notebooks/ 중 어디서 실행해도 경로를 찾도록 설정
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent

assert (ROOT / "src").is_dir(), "저장소 루트 경로를 찾지 못했습니다."
sys.path.insert(0, str(ROOT / "src"))

from loader import load_clean
from classify import add_categories, BIG_CATEGORY_ORDER
from cohort import analyze, MIN_PEERS

print("저장소 위치:", ROOT)

In [ ]:
df = load_clean()

# P1의 loader가 P2 분류를 이미 적용했다면 다시 적용하지 않음
if "big_category" not in df.columns:
    df = add_categories(df)

if "ym" not in df.columns:
    df = df.copy()
    df["ym"] = pd.to_datetime(
        df["transaction_datetime"]
    ).dt.strftime("%Y-%m")

required = {
    "customer_id", "age_group", "gender", "occupation",
    "annual_income", "ym", "big_category", "amount"
}
missing = required - set(df.columns)
assert not missing, f"필수 컬럼 누락: {missing}"

print("거래 건수:", len(df))
print("고객 수:", df["customer_id"].nunique())
print("분석 월:", sorted(df["ym"].unique()))

display(df[
    ["customer_id", "ym", "age_group", "gender",
     "occupation", "annual_income", "big_category", "amount"]
].head())

In [ ]:
MONTH = "2025-06"

active_ids = df.loc[
    df["ym"] == MONTH, "customer_id"
].dropna().unique()

assert len(active_ids) > 0, f"{MONTH} 거래 데이터가 없습니다."

CUSTOMER_ID = str(active_ids[0])
# 예: CUSTOMER_ID = "C001"

result = analyze(df, CUSTOMER_ID, MONTH)

print("고객:", CUSTOMER_ID)
print("분석 월:", MONTH)
print("리포트 문구:", result["message"])
display(pd.DataFrame([result["data"]]))

In [ ]:
profile_cols = [
    "customer_id", "age_group", "gender",
    "occupation", "annual_income"
]

profiles = (
    df[profile_cols]
    .drop_duplicates("customer_id")
    .set_index("customer_id")
)

# 조회한 달에 거래한 고객만 비교 후보
profiles = profiles.loc[active_ids].copy()

def income_band(income):
    if income < 40_000_000:
        return "4천만 원 미만"
    if income < 60_000_000:
        return "4천만~6천만 원"
    return "6천만 원 이상"

profiles["income_band"] = (
    pd.to_numeric(profiles["annual_income"])
    .apply(income_band)
)

stages = [
    ("age_group", "gender", "occupation", "income_band"),
    ("age_group", "gender", "occupation"),
    ("age_group", "gender"),
]

target = profiles.loc[CUSTOMER_ID]
stage_results = []

for stage_number, fields in enumerate(stages, start=1):
    # 비교 고객 본인 제외
    mask = profiles.index != CUSTOMER_ID

    for field in fields:
        mask &= profiles[field].eq(target[field]).fillna(False).to_numpy()

    peer_count = int(mask.sum())

    stage_results.append({
        "단계": stage_number,
        "비교 조건": " × ".join(fields),
        "본인 제외 비교 고객 수": peer_count,
        "20명 충족": peer_count >= MIN_PEERS,
    })

display(pd.DataFrame(stage_results))

In [ ]:
sql_path = ROOT / "sql" / "queries_P4.sql"
assert sql_path.exists(), f"SQL 파일이 없습니다: {sql_path}"

sql = sql_path.read_text(encoding="utf-8")

with sqlite3.connect(":memory:") as conn:
    df[
        ["customer_id", "age_group", "gender", "occupation",
         "annual_income", "ym", "big_category", "amount"]
    ].to_sql(
        "clean_transactions",
        conn,
        index=False,
        if_exists="replace",
    )

    sql_result = pd.read_sql_query(
        sql,
        conn,
        params={"customer_id": CUSTOMER_ID, "month": MONTH},
    )

display(sql_result)

if result["data"]["status"] == "ok":
    assert len(sql_result) == 1, "SQL 결과가 1행이어야 합니다."

    py = result["data"]
    sq = sql_result.iloc[0]

    assert py["cohort_stage"] == int(sq["cohort_stage"])
    assert py["peer_count"] == int(sq["peer_count"])
    assert py["category"] == sq["big_category"]
    assert py["my_spend"] == int(sq["my_spend"])
    assert np.isclose(py["peer_mean"], sq["peer_mean"], atol=0.01)
    assert np.isclose(py["peer_median"], sq["peer_median"], atol=0.01)
    assert np.isclose(
        py["diff_from_median"], sq["diff_from_median"], atol=0.01
    )

    print("검증 PASS: Python과 SQL 결과가 일치합니다.")
else:
    assert sql_result.empty
    print("검증 PASS: 비교 인원 부족으로 양쪽 모두 결과가 없습니다.")